# AI Toolkit by Ostris
## FLUX.1-dev Training


In [ ]:
!nvidia-smi

In [1]:
!git clone https://github.com/ostris/ai-toolkit
!mkdir -p /content/dataset

Cloning into 'ai-toolkit'...
remote: Enumerating objects: 3980, done.
remote: Counting objects: 100% (1969/1969), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 3980 (delta 1852), reused 1753 (delta 1729), pack-reused 2011 (from 1)
Receiving objects: 100% (3980/3980), 29.68 MiB | 30.86 MiB/s, done.
Resolving deltas: 100% (3031/3031), done.
mkdir: /content: Read-only file system


Put your image dataset in the `/content/dataset` folder

In [2]:
!cd ai-toolkit && git submodule update --init --recursive && pip install -r requirements.txt


Submodule 'repositories/batch_annotator' (https://github.com/ostris/batch-annotator) registered for path 'repositories/batch_annotator'
Submodule 'repositories/ipadapter' (https://github.com/tencent-ailab/IP-Adapter.git) registered for path 'repositories/ipadapter'
Submodule 'repositories/leco' (https://github.com/p1atdev/LECO) registered for path 'repositories/leco'
Submodule 'repositories/sd-scripts' (https://github.com/kohya-ss/sd-scripts.git) registered for path 'repositories/sd-scripts'
Cloning into '/Users/mackcesar/PycharmProjects/ai-toolkit/notebooks/ai-toolkit/repositories/batch_annotator'...
Cloning into '/Users/mackcesar/PycharmProjects/ai-toolkit/notebooks/ai-toolkit/repositories/ipadapter'...
Cloning into '/Users/mackcesar/PycharmProjects/ai-toolkit/notebooks/ai-toolkit/repositories/leco'...
Cloning into '/Users/mackcesar/PycharmProjects/ai-toolkit/notebooks/ai-toolkit/repositories/sd-scripts'...
Submodule path 'repositories/batch_annotator': checked out '420e142f6ad3cc14b

## Model License
Training currently only works with FLUX.1-dev. Which means anything you train will inherit the non-commercial license. It is also a gated model, so you need to accept the license on HF before using it. Otherwise, this will fail. Here are the required steps to setup a license.

Sign into HF and accept the model access here [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev)

[Get a READ key from huggingface](https://huggingface.co/settings/tokens/new?) and place it in the next cell after running it.

In [3]:
import getpass
import os

# Prompt for the token
hf_token = getpass.getpass('Enter your HF access token and press enter: ')

# Set the environment variable
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable has been set.")

HF_TOKEN environment variable has been set.


In [4]:
import os
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Setup

This is your config. It is documented pretty well. Normally you would do this as a yaml file, but for colab, this will work. This will run as is without modification, but feel free to edit as you want.

In [22]:
from collections import OrderedDict
import torch

# Detect device and set it appropriately
device = 'mps' if torch.backends.mps.is_built() else ('cuda:0' if torch.cuda.is_available() else 'cpu')

job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        # this name will be the folder and filename name
        ('name', 'gearless_test'),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                # root folder to save training sessions/samples/weights
                ('training_folder', '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164'),
                # uncomment to see performance stats in the terminal every N steps
                #('performance_log_every', 1000),
                ('device', device),  # Updated to be dynamic based on available device
                # if a trigger word is specified, it will be added to captions of training data if it does not already exist
                # alternatively, in your captions you can add [trigger] and it will be replaced with the trigger word
                # ('trigger_word', 'image'),
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', 16),
                    ('linear_alpha', 16)
                ])),
                ('save', OrderedDict([
                    ('dtype', 'float32'),  # Use float32 for MPS compatibility
                    ('save_every', 250),  # save every this many steps
                    ('max_step_saves_to_keep', 4)  # how many intermittent saves to keep
                ])),
                ('datasets', [
                    # datasets are a folder of images. captions need to be txt files with the same name as the image
                    # for instance image2.jpg and image2.txt. Only jpg, jpeg, and png are supported currently
                    # images will automatically be resized and bucketed into the resolution specified
                    OrderedDict([
                        ('folder_path', '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164'),
                        ('caption_ext', 'txt'),
                        ('caption_dropout_rate', 0.05),  # will drop out the caption 5% of time
                        ('shuffle_tokens', False),  # shuffle caption order, split by commas
                        ('cache_latents_to_disk', True),  # leave this true unless you know what you're doing
                        ('resolution', [512, 768, 1024])  # flux enjoys multiple resolutions
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', 1),
                    ('steps', 2000),  # total number of steps to train 500 - 4000 is a good range
                    ('gradient_accumulation_steps', 1),
                    ('train_unet', True),
                    ('train_text_encoder', False),  # probably won't work with flux
                    ('content_or_style', 'balanced'),  # content, style, balanced
                    ('gradient_checkpointing', True),  # need the on unless you have a ton of vram
                    ('noise_scheduler', 'flowmatch'),  # for training only
                    ('optimizer', 'adamw'),  # Use standard optimizer for MPS compatibility
                    ('lr', 1e-4),

                    # uncomment this to skip the pre training sample
                    # ('skip_first_sample', True),

                    # uncomment to completely disable sampling
                    # ('disable_sampling', True),

                    # uncomment to use new vell curved weighting. Experimental but may produce better results
                    # ('linear_timesteps', True),

                    # ema will smooth out learning, but could slow it down. Recommended to leave on.
                    ('ema_config', OrderedDict([
                        ('use_ema', True),
                        ('ema_decay', 0.99)
                    ])),

                    # Use float32 for MPS compatibility
                    ('dtype', 'float32')
                ])),
                ('model', OrderedDict([
                    # huggingface model name or path
                    ('name_or_path', 'black-forest-labs/FLUX.1-dev'),
                    ('is_flux', True),
                    ('quantize', False),  # Disable quantization for MPS compatibility
                    #('low_vram', True),  # uncomment this if the GPU is connected to your monitors. It will use less vram to quantize, but is slower.
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),  # must match train.noise_scheduler
                    ('sample_every', 250),  # sample every this many steps
                    ('width', 1024),
                    ('height', 1024),
                    ('prompts', [
                        # you can add [trigger] to the prompts here and it will be replaced with the trigger word
                        #'[trigger] holding a sign that says \'I LOVE PROMPTS!\'',
                        'woman with red hair, playing chess at the park, bomb going off in the background',
                        'a woman holding a coffee cup, in a beanie, sitting at a cafe',
                        'a horse is a DJ at a night club, fish eye lens, smoke machine, lazer lights, holding a martini',
                        'a man showing off his cool new t shirt at the beach, a shark is jumping out of the water in the background',
                        'a bear building a log cabin in the snow covered mountains',
                        'woman playing the guitar, on stage, singing a song, laser lights, punk rocker',
                        'hipster man with a beard, building a chair, in a wood shop',
                        'photo of a man, white background, medium shot, modeling clothing, studio lighting, white backdrop',
                        'a man holding a sign that says, \'this is a sign\'',
                        'a bulldog, in a post apocalyptic world, with a shotgun, in a leather jacket, in a desert, with a motorcycle'
                    ]),
                    ('neg', ''),  # not used on flux
                    ('seed', 42),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', 20)
                ]))
            ])
        ])
    ])),
    # you can add any additional meta info here. [name] is replaced with config name at top
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])

## Run it

Below does all the magic. Check your folders to the left. Items will be in output/LoRA/your_name_v1 In the samples folder, there are preiodic sampled. This doesnt work great with colab. They will be in /content/output

In [23]:
run_job(job_to_run)


Job function is running.
Job configuration: OrderedDict({'job': 'extension', 'config': OrderedDict({'name': 'gearless_test', 'process': [OrderedDict({'type': 'sd_trainer', 'training_folder': '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164', 'device': 'mps', 'network': OrderedDict({'type': 'lora', 'linear': 16, 'linear_alpha': 16}), 'save': OrderedDict({'dtype': 'float32', 'save_every': 250, 'max_step_saves_to_keep': 4}), 'datasets': [OrderedDict({'folder_path': '/Users/mackcesar/PycharmProjects/ai-toolkit/datasets/0b541f3d-b862-4e2a-9b38-1781274d9164', 'caption_ext': 'txt', 'caption_dropout_rate': 0.05, 'shuffle_tokens': False, 'cache_latents_to_disk': True, 'resolution': [512, 768, 1024]})], 'train': OrderedDict({'batch_size': 1, 'steps': 2000, 'gradient_accumulation_steps': 1, 'train_unet': True, 'train_text_encoder': False, 'content_or_style': 'balanced', 'gradient_checkpointing': True, 'noise_scheduler': 'flowmatch', 'optimizer': 'adamw', 

## Done

Check your ourput dir and get your slider
